# 02 - Clean and feature engineer

Type coercion and the derived fields we lean on everywhere downstream. Outputs go to `data/processed/` as parquet so notebooks 03+ skip the slow CSV reads.

Two parts: items + shipments features first, then order-level rollup and the customer first-delivered-order cohort used by Task B2.

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

DATA_DIR = Path('..').resolve().parent
OUT_DIR  = Path('..') / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / 'customers.csv')
sellers   = pd.read_csv(DATA_DIR / 'sellers.csv')
products  = pd.read_csv(DATA_DIR / 'products.csv')
orders    = pd.read_csv(DATA_DIR / 'orders.csv')
items     = pd.read_csv(DATA_DIR / 'order_items.csv')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv')

orders['created_at']             = pd.to_datetime(orders['created_at'])
orders['promised_delivery_date'] = pd.to_datetime(orders['promised_delivery_date'])
shipments['shipped_at']          = pd.to_datetime(shipments['shipped_at'])
shipments['delivered_at']        = pd.to_datetime(shipments['delivered_at'])
customers['signup_date']         = pd.to_datetime(customers['signup_date'])

len(customers), len(orders), len(items), len(shipments)

(25000, 100000, 169929, 91994)

## Item-level features

`gmv_line` follows the brief literally: `quantity * unit_price`, before any platform fee or discount adjustment. We also keep a discount-adjusted figure as a side metric, but the headline GMV stays gross per the assessment definition.

In [20]:
items_enriched = items.copy()
items_enriched['gmv_line']       = items_enriched['quantity'] * items_enriched['unit_price']
items_enriched['gmv_after_disc'] = items_enriched['gmv_line'] * (1 - items_enriched['discount_pct'])
items_enriched['platform_fee']   = items_enriched['gmv_line'] * items_enriched['platform_fee_pct']

items_enriched[['quantity', 'unit_price', 'discount_pct', 'platform_fee_pct',
                'gmv_line', 'gmv_after_disc', 'platform_fee']].describe().round(2)

,quantity,unit_price,discount_pct,platform_fee_pct,gmv_line,gmv_after_disc,platform_fee
count,169929.00,169929.00,169929.00,169929.00,169929.00,169929.00,169929.00
mean,1.38,14486.02,0.05,0.12,19961.24,18958.97,1887.93
std,0.63,20371.69,0.08,0.03,32103.22,30610.11,2658.96
min,1.00,26.00,0.00,0.06,26.00,20.80,2.86
25%,1.00,1525.00,0.00,0.09,1818.00,1729.00,213.50
50%,1.00,4829.00,0.00,0.12,5918.00,5590.00,800.02
75%,2.00,18566.00,0.09,0.14,22886.00,21811.00,2553.60
max,3.00,91741.00,0.25,0.17,273654.00,272067.00,27206.70


## Shipment-level features

Three fields we will use repeatedly:
- `delivery_delay_days`: actual delivery date minus promised date, in whole days. NaN for Lost or still-in-transit shipments.
- `is_delayed`: True for any non-OnTime outcome, treating Lost as a failed promise. NaN for InTransit since the outcome is not yet known.
- `lane`: ship_from_city -> ship_to_city, handy for groupby on routes.

In [21]:
ships = shipments.merge(orders[['order_id', 'promised_delivery_date']], on='order_id', how='left')

delivered_date = ships['delivered_at'].dt.normalize()
promised_date  = ships['promised_delivery_date'].dt.normalize()
ships['delivery_delay_days'] = (delivered_date - promised_date).dt.days

status = ships['delivery_status']
is_delayed = pd.Series(pd.NA, index=ships.index, dtype='boolean')
is_delayed[status == 'OnTime'] = False
is_delayed[status.isin(['Late_1_2d', 'Late_3_5d', 'Late_5p', 'Lost'])] = True
ships['is_delayed'] = is_delayed

ships['lane']            = ships['ship_from_city'] + ' -> ' + ships['ship_to_city']
ships['delivered_month'] = ships['delivered_at'].dt.to_period('M').astype(str)
ships.loc[ships['delivered_month'] == 'NaT', 'delivered_month'] = pd.NA

shipments_enriched = ships.drop(columns=['promised_delivery_date'])

shipments_enriched.head(3)

,shipment_id,order_id,carrier,shipped_at,delivered_at,ship_from_city,ship_to_city,shipping_cost,delivery_status,delivery_delay_days,is_delayed,lane,delivered_month
0,SH000001,ORD000001,Ekart,2024-07-01 13:00:27,2024-07-02 18:41:52,Mumbai,Delhi,95.33,OnTime,-1.0,False,Mumbai -> Delhi,2024-07
1,SH000002,ORD000002,Delhivery,2024-07-01 16:17:32,2024-07-03 04:35:14,Delhi,Kochi,109.57,Late_1_2d,0.0,True,Delhi -> Kochi,2024-07
2,SH000003,ORD000003,Delhivery,2024-07-01 14:39:14,2024-07-02 21:48:54,Jaipur,Kolkata,77.16,OnTime,-1.0,False,Jaipur -> Kolkata,2024-07


In [22]:
print('delay days summary (delivered shipments only):')
print(shipments_enriched['delivery_delay_days'].describe().round(2))
print()
print('is_delayed by delivery_status:')
print(pd.crosstab(shipments_enriched['delivery_status'],
                  shipments_enriched['is_delayed'].astype('object'),
                  dropna=False))

delay days summary (delivered shipments only):
count    86521.00
mean        -1.22
std          1.43
min         -5.00
25%         -2.00
50%         -1.00
75%          0.00
max          5.00
Name: delivery_delay_days, dtype: float64

is_delayed by delivery_status:
is_delayed       False  True 
delivery_status              
Late_1_2d            0  20388
Late_3_5d            0   2424
Late_5p              0     27
Lost                 0    430
OnTime           63682      0


## Order-level rollup

Aggregate items up to one row per order, then join with customers and the order's shipment into a single wide `orders_enriched` table.

In [23]:
order_agg = items_enriched.groupby('order_id').agg(
    order_gmv            = ('gmv_line', 'sum'),
    order_gmv_after_disc = ('gmv_after_disc', 'sum'),
    order_platform_fee   = ('platform_fee', 'sum'),
    order_items_count    = ('order_item_id', 'count'),
    order_units          = ('quantity', 'sum'),
    order_sellers_count  = ('seller_id', 'nunique'),
).reset_index()

order_agg.head()

,order_id,order_gmv,order_gmv_after_disc,order_platform_fee,order_items_count,order_units,order_sellers_count
0,ORD000001,1496.0,1496.00,224.40,1,2,1
1,ORD000002,53361.0,53361.00,4268.88,1,1,1
2,ORD000003,5826.0,5826.00,757.38,1,3,1
3,ORD000004,27644.0,26305.67,3721.18,2,2,2
4,ORD000005,132215.0,131882.38,11962.82,4,5,4


Each non-cancelled order should have exactly one shipment. Worth verifying before we collapse shipments down by order_id.

In [24]:
sh_per_order = shipments_enriched.groupby('order_id').size()
print('shipments per order distribution:')
print(sh_per_order.value_counts())
assert sh_per_order.max() == 1, 'some orders have multiple shipments — need to aggregate'

shipments per order distribution:
1    91994
dtype: int64


## Build orders_enriched

In [25]:
ship_cols = ['order_id', 'carrier', 'shipped_at', 'delivered_at',
             'ship_from_city', 'ship_to_city', 'shipping_cost',
             'delivery_status', 'delivery_delay_days', 'is_delayed',
             'lane', 'delivered_month']

orders_enriched = (orders
    .merge(customers[['customer_id', 'city', 'state', 'segment', 'signup_date']],
           on='customer_id', how='left')
    .merge(order_agg, on='order_id', how='left')
    .merge(shipments_enriched[ship_cols], on='order_id', how='left')
)

orders_enriched['order_month']   = orders_enriched['created_at'].dt.to_period('M').astype(str)
orders_enriched['promised_days'] = (orders_enriched['promised_delivery_date'].dt.normalize()
                                    - orders_enriched['created_at'].dt.normalize()).dt.days

orders_enriched.shape, orders_enriched.columns.tolist()

((100000, 30),
 ['order_id',
  'customer_id',
  'created_at',
  'status',
  'payment_method',
  'promised_delivery_date',
  'is_fast_delivery_eligible',
  'city',
  'state',
  'segment',
  'signup_date',
  'order_gmv',
  'order_gmv_after_disc',
  'order_platform_fee',
  'order_items_count',
  'order_units',
  'order_sellers_count',
  'carrier',
  'shipped_at',
  'delivered_at',
  'ship_from_city',
  'ship_to_city',
  'shipping_cost',
  'delivery_status',
  'delivery_delay_days',
  'is_delayed',
  'lane',
  'delivered_month',
  'order_month',
  'promised_days'])

In [26]:
print('orders_enriched shape  :', orders_enriched.shape)
print('null order_gmv         :', orders_enriched['order_gmv'].isna().sum(), '(expected: 0)')
print('null delivery_status   :', orders_enriched['delivery_status'].isna().sum(), '(expected: 8006 cancelled)')
print()
print('order status vs has_shipment:')
print(pd.crosstab(orders_enriched['status'], orders_enriched['delivery_status'].isna()))
print()
print('promised_days distribution (fast vs not):')
print(orders_enriched.groupby('is_fast_delivery_eligible')['promised_days'].value_counts().head(10))

orders_enriched shape  : (100000, 30)
null order_gmv         : 0 (expected: 0)
null delivery_status   : 8006 (expected: 8006 cancelled)

order status vs has_shipment:
delivery_status  False  True 
status                       
Cancelled            0   8006
Delivered        82069      0
Returned          4882      0
Shipped           5043      0

promised_days distribution (fast vs not):
is_fast_delivery_eligible  promised_days
False                      3                11764
                           5                11717
                           4                11429
True                       2                65090
Name: promised_days, dtype: int64


## Customer first delivered order

Task B2 asks for the 90-day repeat rate split by whether each customer's first delivered order was on-time or delayed. We tag the cohort here so the SQL and Python paths share the same definition.

Definition: first delivered order = the order with the earliest `delivered_at` per customer. We exclude InTransit and Lost (no `delivered_at`) and orders with status not in (Delivered, Returned).

In [27]:
delivered = orders_enriched[orders_enriched['delivered_at'].notna()].copy()

first_delivered = (delivered
    .sort_values(['customer_id', 'delivered_at'])
    .groupby('customer_id', as_index=False)
    .first()[['customer_id', 'order_id', 'delivered_at', 'delivery_status', 'is_delayed']]
    .rename(columns={
        'order_id': 'first_order_id',
        'delivered_at': 'first_delivered_at',
        'delivery_status': 'first_delivery_status',
        'is_delayed': 'first_is_delayed',
    })
)

first_delivered['first_order_delay_status'] = np.where(
    first_delivered['first_is_delayed'] == True, 'Delayed',
    np.where(first_delivered['first_is_delayed'] == False, 'OnTime', 'Unknown')
)

print('customers with at least one delivered order:', len(first_delivered))
print()
print('first_order_delay_status:')
print(first_delivered['first_order_delay_status'].value_counts(dropna=False))
print()
print('first_delivery_status:')
print(first_delivered['first_delivery_status'].value_counts(dropna=False))

customers with at least one delivered order: 23443

first_order_delay_status:
OnTime     17439
Delayed     6004
Name: first_order_delay_status, dtype: int64

first_delivery_status:
OnTime       17439
Late_1_2d     5350
Late_3_5d      646
Late_5p          8
Name: first_delivery_status, dtype: int64


## Save

In [28]:
items_enriched.to_parquet(OUT_DIR / 'items_enriched.parquet', index=False)
shipments_enriched.to_parquet(OUT_DIR / 'shipments_enriched.parquet', index=False)
orders_enriched.to_parquet(OUT_DIR / 'orders_enriched.parquet', index=False)
first_delivered.to_parquet(OUT_DIR / 'customer_first_order.parquet', index=False)

for fname in ['items_enriched.parquet', 'shipments_enriched.parquet',
              'orders_enriched.parquet', 'customer_first_order.parquet']:
    p = OUT_DIR / fname
    print(f'{fname:30s} {p.resolve()}  ({p.stat().st_size/1024:.1f} KB)')

items_enriched.parquet         L:\OJ Commerece assessment\submission\data\processed\items_enriched.parquet  (5303.7 KB)
shipments_enriched.parquet     L:\OJ Commerece assessment\submission\data\processed\shipments_enriched.parquet  (3399.1 KB)
orders_enriched.parquet        L:\OJ Commerece assessment\submission\data\processed\orders_enriched.parquet  (5571.3 KB)
customer_first_order.parquet   L:\OJ Commerece assessment\submission\data\processed\customer_first_order.parquet  (554.2 KB)


## Reconciliation

Cheap checks that protect every downstream notebook.

In [29]:
# 1. GMV from items must equal GMV summed from orders_enriched
gmv_items  = items_enriched['gmv_line'].sum()
gmv_orders = orders_enriched['order_gmv'].sum()
assert abs(gmv_items - gmv_orders) < 1e-6, f'gmv mismatch: items {gmv_items} != orders {gmv_orders}'

# 2. Cancelled orders must have no shipment fields
cancelled_with_ship = ((orders_enriched['status'] == 'Cancelled') & orders_enriched['delivery_status'].notna()).sum()
assert cancelled_with_ship == 0, 'cancelled orders should have no shipment'

# 3. Customers with >=1 delivered order must equal first_delivered row count
unique_cust_delivered = orders_enriched.loc[orders_enriched['delivered_at'].notna(), 'customer_id'].nunique()
assert len(first_delivered) == unique_cust_delivered, 'first_delivered count mismatch'

# 4. is_delayed totals stay consistent
late_or_lost = shipments_enriched['delivery_status'].isin(['Late_1_2d', 'Late_3_5d', 'Late_5p', 'Lost']).sum()
assert late_or_lost == (shipments_enriched['is_delayed'] == True).sum()

print('gmv (items)              :', round(gmv_items, 2))
print('gmv (orders rollup)      :', round(gmv_orders, 2))
print('orders                   :', len(orders_enriched))
print('  delivered              :', (orders_enriched['status'] == 'Delivered').sum())
print('  returned               :', (orders_enriched['status'] == 'Returned').sum())
print('  shipped (in-transit)   :', (orders_enriched['status'] == 'Shipped').sum())
print('  cancelled              :', (orders_enriched['status'] == 'Cancelled').sum())
print('customers w/ delivered   :', len(first_delivered))

gmv (items)              : 3391994349.0
gmv (orders rollup)      : 3391994349.0
orders                   : 100000
  delivered              : 82069
  returned               : 4882
  shipped (in-transit)   : 5043
  cancelled              : 8006
customers w/ delivered   : 23443
